# Step 1 — Create disposable email + Submit to medRxiv
**YOPmail** — zero signup, no CAPTCHA, no phone. Inbox lives 8 days.

In [ ]:
# ── Install ────────────────────────────────────────────────────────────────
!pip install playwright -q
!playwright install chromium
!apt-get install -y -q libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 \
  libnss3 libatk1.0-0 libatk-bridge2.0-0 libdrm2 libxkbcommon0 libasound2 \
  libxshmfence1 libpango-1.0-0 libcairo2 libcups2 libdbus-1-3 libexpat1 \
  libfontconfig1 libglib2.0-0 libnspr4 libx11-6 libx11-xcb1 libxcb1 \
  libxext6 libxrender1 libxtst6
print('✅ Ready')

In [ ]:
# ── Create YOPmail address (zero signup) ───────────────────────────────────
from playwright.async_api import async_playwright
from IPython.display import Image, display
import random, string

# Generate a unique address
suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
EMAIL = f'penux.research.{suffix}@yopmail.com'
print(f'📧 Your new email: {EMAIL}')
print('   (no signup needed — inbox is ready immediately)')

async def check_inbox():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox','--disable-dev-shm-usage','--disable-gpu']
        )
        ctx = await browser.new_context(
            ignore_https_errors=True,
            viewport={'width':1280,'height':900}
        )
        page = await ctx.new_page()
        username = EMAIL.split('@')[0]
        await page.goto(f'https://yopmail.com/en/wm?login={username}', wait_until='networkidle', timeout=30000)
        await page.screenshot(path='/tmp/yopmail_inbox.png', full_page=False)
        display(Image('/tmp/yopmail_inbox.png'))
        print(f'✅ Inbox ready at: https://yopmail.com/en/wm?login={username}')
        await browser.close()

await check_inbox()

In [ ]:
# ── Register on medRxiv with new email ────────────────────────────────────
PASSWORD = 'PenuX2026!'

TITLE = (
    'PenuX: A Comparative Study of 11 Machine Learning and Deep Learning Models '
    'for Early Severity Prediction of Acute Pancreatitis Using Routine Admission '
    'Laboratory Values, with FHIR R4 Integration'
)
ABSTRACT = """Background: Severe Acute Pancreatitis (SAP) carries a mortality rate of 20-30% and requires early risk stratification. Classical scoring systems (Ranson, BISAP, APACHE II) require 24-48 hours of serial laboratory observation and lack EHR integration.
Methods: Retrospective analysis of 722 AP admissions (585 severe/137 mild; Atlanta 2012) from a single Chinese institution. Eleven models trained on 106 admission laboratory features using 5-fold stratified cross-validation.
Results: Random Forest achieved AUC=0.877, sensitivity=96.8% at threshold 0.535. CNN-LSTM: AUC=0.772. Key predictors: calcium, D-dimer, LDH, lactate, hematocrit. Label inversion effect identified.
Conclusions: Random Forest achieves SAP triage from a single admission blood draw, eliminating the 24-48h observation window. Open-source platform with FHIR R4 integration at https://penux.uk"""

import urllib.request, os
DOCX_URL = 'https://raw.githubusercontent.com/netanelcyber/penuX/claude/pensive-pascal-a0l7a8/cureus_submission/manuscript_medrxiv.docx'
urllib.request.urlretrieve(DOCX_URL, '/tmp/manuscript_penux.docx')
print(f'✅ Manuscript downloaded: {os.path.getsize("/tmp/manuscript_penux.docx"):,} bytes')
print(f'✅ Will register with: {EMAIL}')

In [ ]:
# ── Register + submit to medRxiv ──────────────────────────────────────────
async def register_and_submit():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox','--disable-dev-shm-usage','--disable-gpu']
        )
        ctx = await browser.new_context(
            ignore_https_errors=True,
            viewport={'width':1280,'height':900},
            user_agent='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
        )
        page = await ctx.new_page()

        # ── 1. Register ────────────────────────────────────────────────────
        print('Step 1: Loading medRxiv registration...')
        await page.goto('https://submit.medrxiv.org/register', wait_until='networkidle', timeout=30000)
        await page.screenshot(path='/tmp/reg1.png')
        display(Image('/tmp/reg1.png'))
        print(f'URL: {page.url}')

        # Fill registration form
        fields = {
            'input[name="first_name"], #first_name': 'Netanel',
            'input[name="last_name"], #last_name': 'Shoshany',
            'input[name="email"], input[type="email"], #email': EMAIL,
            'input[name="password"], input[type="password"], #password': PASSWORD,
            'input[name="password_confirmation"], #password_confirmation': PASSWORD,
        }
        for selectors, value in fields.items():
            for sel in selectors.split(', '):
                try:
                    await page.fill(sel, value, timeout=2000)
                    print(f'  ✅ {sel.split("[")[0]} filled')
                    break
                except: pass

        await page.screenshot(path='/tmp/reg2_filled.png')
        display(Image('/tmp/reg2_filled.png'))

        try:
            await page.click('button[type="submit"], input[type="submit"]', timeout=5000)
            await page.wait_for_load_state('networkidle', timeout=15000)
            await page.screenshot(path='/tmp/reg3_after.png')
            display(Image('/tmp/reg3_after.png'))
            print(f'After registration URL: {page.url}')
        except Exception as e:
            print(f'Register submit: {e}')

        # ── 2. Check YOPmail for verification link ─────────────────────────
        print('\nStep 2: Checking YOPmail for verification email...')
        import asyncio
        await asyncio.sleep(5)
        username = EMAIL.split('@')[0]
        await page.goto(f'https://yopmail.com/en/wm?login={username}', wait_until='networkidle', timeout=30000)
        await page.screenshot(path='/tmp/yop_after_reg.png')
        display(Image('/tmp/yop_after_reg.png'))

        # Try to find and click verification link
        try:
            # Look in iframe (YOPmail renders emails in iframe)
            frame = page.frame_locator('#ifmail').first
            verify_link = await frame.locator('a[href*="confirm"], a[href*="verif"], a[href*="activ"]').first.get_attribute('href')
            if verify_link:
                print(f'✅ Found verification link: {verify_link[:80]}...')
                await page.goto(verify_link, wait_until='networkidle', timeout=20000)
                await page.screenshot(path='/tmp/verified.png')
                display(Image('/tmp/verified.png'))
                print(f'After verification URL: {page.url}')
            else:
                print('⚠️  No verification link found yet — check YOPmail manually')
        except Exception as e:
            print(f'Verification: {e}')

        await browser.close()

await register_and_submit()

In [ ]:
# ── Submit manuscript after account verified ──────────────────────────────
async def submit():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox','--disable-dev-shm-usage','--disable-gpu']
        )
        ctx = await browser.new_context(
            ignore_https_errors=True,
            viewport={'width':1280,'height':900},
            user_agent='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
        )
        page = await ctx.new_page()

        # Sign in
        await page.goto('https://submit.medrxiv.org', wait_until='networkidle', timeout=30000)
        try:
            await page.click('text=Log In', timeout=3000)
            await page.wait_for_load_state('networkidle')
        except:
            await page.goto('https://submit.medrxiv.org/login', wait_until='networkidle', timeout=20000)

        for sel in ['input[name="email"]','input[type="email"]','#email']:
            try: await page.fill(sel, EMAIL, timeout=2000); break
            except: pass
        for sel in ['input[name="password"]','input[type="password"]','#password']:
            try: await page.fill(sel, PASSWORD, timeout=2000); break
            except: pass
        await page.click('button[type="submit"],input[type="submit"]', timeout=5000)
        await page.wait_for_load_state('networkidle', timeout=15000)
        print(f'Signed in. URL: {page.url}')
        await page.screenshot(path='/tmp/submit_loggedin.png')
        display(Image('/tmp/submit_loggedin.png'))

        # New submission
        try:
            await page.click('text=New Submission', timeout=5000)
            await page.wait_for_load_state('networkidle')
        except:
            await page.goto('https://submit.medrxiv.org/submit', wait_until='networkidle', timeout=20000)

        await page.screenshot(path='/tmp/submit_form.png')
        display(Image('/tmp/submit_form.png'))

        # Fill form
        for sel in ['input[name="title"]','#title']:
            try: await page.fill(sel, TITLE, timeout=3000); break
            except: pass
        for sel in ['textarea[name="abstract"]','#abstract']:
            try: await page.fill(sel, ABSTRACT, timeout=3000); break
            except: pass
        try:
            await page.locator('input[type="file"]').first.set_input_files('/tmp/manuscript_penux.docx')
            print('✅ File uploaded')
        except Exception as e:
            print(f'⚠️ File upload: {e}')

        await page.screenshot(path='/tmp/submit_filled.png')
        display(Image('/tmp/submit_filled.png'))

        # Submit
        try:
            await page.click('button:has-text("Submit"),input[type="submit"]', timeout=8000)
            await page.wait_for_load_state('networkidle', timeout=30000)
            await page.screenshot(path='/tmp/submit_done.png')
            display(Image('/tmp/submit_done.png'))
            print(f'✅ SUBMITTED! URL: {page.url}')
        except Exception as e:
            await page.screenshot(path='/tmp/submit_err.png')
            display(Image('/tmp/submit_err.png'))
            print(f'⚠️ Submit: {e}')

        await browser.close()

await submit()